# Offline audit of a saved docking result

This notebook reads a retained minimal native Vina software control. Its raw PDBQT
inputs are **unassessed**: internal agreement does not establish chemical validity,
predictive accuracy, authentication or a scientific benchmark. The audit does not
run docking or perform molecular/viewer operations. See `result_audit.md` for its
contract and boundaries.

In [1]:
import base64
import json
from collections import Counter
from copy import deepcopy
from pathlib import Path

import dockingmt as dmt

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "devguide/validation/data/audit/minimal_vina_result.json").is_file())
record = json.loads((root / "devguide/validation/data/audit/minimal_vina_result.json").read_text())
print(json.dumps(record["fixture_evidence"], indent=2))

{
  "purpose": "Minimal native Vina software control; raw PDBQT inputs are unassessed. No chemical or predictive qualification.",
  "source": "tests/test_engines.py minimal raw PDBQT constants",
  "recorded": "2026-10-03"
}


## Complete internal agreement

The supported API accepts an already loaded record, returning detached JSON data.
The caller owns file access. Both captured inputs have matching recorded SHA-256
hashes; score context and ranking observations agree with the retained claims.

In [2]:
report = dmt.audit_result(record)
assert report["status"] == "consistent"
assert json.loads(json.dumps(report, allow_nan=False)) == report
print(json.dumps({"status": report["status"],
                  "checks": dict(Counter(check["status"] for check in report["checks"])),
                  "captured_sizes_bytes": dmt.verify_captured_inputs(record["provenance"]["backend_artifacts"])}, indent=2))

{
  "status": "consistent",
  "checks": {
    "consistent": 164
  },
  "captured_sizes_bytes": {
    "receptor": 317,
    "partner": 181
  }
}


## Missing evidence

Removing retained bytes leaves the digest claim available but unverifiable. Missing
evidence produces `incomplete`; no source input is reopened or silently recreated.

In [3]:
incomplete = deepcopy(record)
del incomplete["provenance"]["backend_artifacts"]["partner"]["content_base64"]
missing_report = dmt.audit_result(incomplete)
assert missing_report["status"] == "incomplete"
print(json.dumps([check for check in missing_report["checks"]
                  if check["status"] != "consistent"], indent=2))

[
  {
    "check": "captured_input",
    "path": "/provenance/backend_artifacts/partner",
    "status": "incomplete",
    "reason": "The partner PDBQT input bytes were not captured."
  }
]


## Contradictory bytes and stale ranking policy

These copies deliberately introduce contradictions. The first changes bytes while
keeping the recorded digest. The second changes the latest policy without recording
a new ranking. Calling `rank_by` instead retains a coherent additional decision.

In [4]:
changed = deepcopy(record)
changed["provenance"]["backend_artifacts"]["partner"]["content_base64"] = base64.b64encode(b"changed bytes").decode()
changed_report = dmt.audit_result(changed)
assert changed_report["status"] == "inconsistent"
ranked = dmt.DockingResult.from_dict(record).rank_by("inter", ascending=False).to_dict()
assert dmt.audit_result(ranked)["status"] == "consistent"
ranked["provenance"]["ranking_policy"]["ascending"] = True
stale_report = dmt.audit_result(ranked)
assert stale_report["status"] == "inconsistent"
print(json.dumps({"changed_bytes": [check for check in changed_report["checks"] if check["status"] == "inconsistent"],
                  "stale_policy": [check for check in stale_report["checks"] if check["status"] == "inconsistent"]}, indent=2))

{
  "changed_bytes": [
    {
      "check": "captured_input",
      "path": "/provenance/backend_artifacts/partner",
      "status": "inconsistent",
      "reason": "The partner PDBQT input bytes failed SHA-256 validation."
    }
  ],
  "stale_policy": [
    {
      "check": "ranking_policy",
      "path": "/provenance/ranking_policy",
      "status": "inconsistent",
      "reason": "claims_disagree"
    }
  ]
}
